# 02 - Evaluation: Baseline on the Test Set

Evaluates the baseline trained in `01_training.ipynb`. **No training happens here** - the
checkpoint is loaded and measured, so results are reproducible.

This notebook produces the numbers every later phase must be compared against:

| phase | compared here |
|---|---|
| Knowledge Distillation (student) | test top-1 / top-5 |
| TensorRT engine | latency + accuracy after optimization |
| Jetson Orin Nano | real-time FPS on the edge device |

Measured: Top-1 / Top-5 accuracy, per-class metrics, confusion matrix, representative
correct/incorrect predictions, parameter count, checkpoint size, and single-image latency.

Requirements: `models/baseline/best.pt` (from 01) and the test split of the dataset used
in 01, found through the same `DATASET_ROOT` mechanism. The distilled student from
`04_distillation.ipynb` is evaluated the same way: point `CHECKPOINT_PATH` at
`models/distilled/best.pt` - no code changes.

## 1. Environment

In [ ]:
import os, sys, time, random, textwrap

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset

%matplotlib inline

print("Python        :", sys.version.split()[0])
print("PyTorch       :", torch.__version__)
print("torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device        :", DEVICE, "(seed =", SEED, ")")

## 2. Configuration

Same environment-variable override pattern as 01 - no machine-specific paths.

In [ ]:
from pathlib import Path

DATASET_ROOT    = Path(os.environ.get("STANFORD_CARS_ROOT", "../data/stanford_cars"))
CHECKPOINT_PATH = Path(os.environ.get("CHECKPOINT_PATH", "../models/baseline/best.pt"))

BATCH_SIZE  = 32
NUM_WORKERS = 2

LATENCY_WARMUP    = 20    # warmup iterations before timing
LATENCY_ITERS_GPU = 200   # timed iterations for GPU latency
LATENCY_ITERS_CPU = 50    # timed iterations for CPU latency (CPU is slow)

print("DATASET_ROOT    :", DATASET_ROOT)
print("CHECKPOINT_PATH :", CHECKPOINT_PATH)

## 3. Load checkpoint

Everything required to rebuild inference lives inside the checkpoint written by 01:
architecture name, weights, class mapping, and preprocessing. The notebook therefore has
**no hidden assumptions** beyond the checkpoint itself - and it equally accepts the
distilled student checkpoint from 04, whose architecture is recorded the same way.

The dataset helpers below are repeated verbatim from `01_training.ipynb` because notebooks
share no state - each one must be runnable independently.

In [ ]:
def discover_stanford_cars(root):
    """Find a Stanford Cars layout under `root`.

    Returns {"classes": [str], "train": [(path, label)], "test": [(path, label)]}
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            "DATASET_ROOT does not exist: " + str(root)
            + "\nSet the STANFORD_CARS_ROOT environment variable or edit the configuration cell.")
    if (root / "cars_meta.mat").exists() and (root / "cars_annos.mat").exists():
        return _discover_devkit(root)
    if (root / "train").is_dir() and (root / "test").is_dir():
        return _discover_class_folders(root)
    raise FileNotFoundError(
        "No supported Stanford Cars layout under " + str(root) + "\n"
        + "Layout A (devkit):  cars_meta.mat, cars_annos.mat, cars_train/, cars_test/\n"
        + "Layout B (folders): train/<class name>/*.jpg, test/<class name>/*.jpg\n"
        + "Download the dataset manually (e.g. the Kaggle mirror "
        + "jutrera/stanford-car-dataset-by-classes-folder).")


def _discover_devkit(root):
    from scipy.io import loadmat  # only needed for this layout
    class_names = [str(c[0]) for c in loadmat(root / "cars_meta.mat")["class_names"].ravel()]
    train_samples, test_samples = [], []
    for a in loadmat(root / "cars_annos.mat")["annotations"].ravel():
        rel     = str(np.asarray(a["relative_im_path"]).ravel()[0])
        label   = int(np.asarray(a["class"]).ravel()[0]) - 1
        is_test = bool(int(np.asarray(a["test"]).ravel()[0]))
        (test_samples if is_test else train_samples).append((root / rel, label))
    return {"classes": class_names, "train": train_samples, "test": test_samples}


def _discover_class_folders(root):
    classes = sorted(p.name for p in (root / "train").iterdir() if p.is_dir())
    class_to_idx = {name: i for i, name in enumerate(classes)}
    result = {"classes": classes, "train": [], "test": []}
    for split in ("train", "test"):
        for class_dir in sorted((root / split).iterdir()):
            if not class_dir.is_dir():
                continue
            label = class_to_idx[class_dir.name]
            for pattern in ("*.jpg", "*.jpeg", "*.png"):
                for img_path in sorted(class_dir.glob(pattern)):
                    result[split].append((img_path, label))
    return result


class StanfordCarsDataset(Dataset):
    """Dataset over (path, label) pairs; transform is passed in so train and
    eval copies can use different preprocessing over the same images."""

    def __init__(self, samples, classes, transform=None):
        self.samples = samples
        self.classes = classes
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)  # our own checkpoint

print("checkpoint    :", CHECKPOINT_PATH)
print("architecture  :", ckpt["model_arch"])
print("num classes   :", ckpt["num_classes"])
print("saved at      :", ckpt.get("saved_at", "n/a"), "| torch", ckpt.get("torch_version", "n/a"))
print("training cfg  :", ckpt["training_config"])
print("val metrics   :", ckpt["metrics"])
print()

# Rebuild the model from the architecture recorded in the checkpoint
# (efficientnet_b0 baseline from 01, or the mobilenet_v3_small student from 04).
def build_model_from_checkpoint(ckpt):
    arch = ckpt["model_arch"]
    if arch == "efficientnet_b0":
        from torchvision.models import efficientnet_b0
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, ckpt["num_classes"])
    elif arch == "mobilenet_v3_small":
        from torchvision.models import mobilenet_v3_small
        model = mobilenet_v3_small(weights=None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, ckpt["num_classes"])
    else:
        raise ValueError("unknown architecture: " + str(arch))
    return model


model = build_model_from_checkpoint(ckpt)
model.load_state_dict(ckpt["state_dict"])
model.to(DEVICE).eval()

# Rebuild preprocessing exactly as 01 configured it.
eval_tf = build_transforms(ckpt["preprocessing"], augment=False)
print("preprocessing :", ckpt["preprocessing"])
print()

# Load the test split; the class order must match the checkpoint.
data = discover_stanford_cars(DATASET_ROOT)
if data["classes"] != ckpt["classes"]:
    raise ValueError(
        "class order of the dataset does not match the checkpoint - "
        "make sure DATASET_ROOT points at the same dataset layout used by 01_training.ipynb")
test_ds = StanfordCarsDataset(data["test"], ckpt["classes"], transform=eval_tf)
print("test samples  :", len(test_ds))

## 4. Test-set evaluation

Run the checkpoint over the full official test split (8,041 images) and keep every
prediction for the per-class analysis below.

In [ ]:
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")

criterion = nn.CrossEntropyLoss()
all_labels, all_preds, all_confs = [], [], []
total_loss, top5_hits, n = 0.0, 0, 0
t0 = time.time()

model.eval()
with torch.no_grad():
    for images, targets in test_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        outputs = model(images)
        total_loss += criterion(outputs, targets).item() * targets.size(0)
        top5_hits += (outputs.topk(5, dim=1).indices == targets[:, None]).any(dim=1).sum().item()
        probs = torch.softmax(outputs, dim=1)
        conf, pred = probs.max(dim=1)
        all_labels.append(targets.cpu().numpy())
        all_preds.append(pred.cpu().numpy())
        all_confs.append(conf.cpu().numpy())
        n += targets.size(0)

labels = np.concatenate(all_labels)
preds  = np.concatenate(all_preds)
confs  = np.concatenate(all_confs)
num_classes = len(ckpt["classes"])

top1 = (labels == preds).mean() * 100.0
top5 = top5_hits / n * 100.0
eval_seconds = time.time() - t0

print("test images      : %d" % n)
print("test loss        : %.4f" % (total_loss / n))
print("test top-1       : %.2f%%" % top1)
print("test top-5       : %.2f%%" % top5)
print("mean confidence  : %.1f%%" % (confs.mean() * 100.0))
print("evaluation time  : %.1fs (%.0f img/s, batch=%d)" % (eval_seconds, n / eval_seconds, BATCH_SIZE))

## 5. Confusion matrix

196 classes make a fully labelled matrix unreadable, so we plot it as a heatmap (diagonal =
correct predictions) and then list the class pairs the model confuses most.

In [ ]:
cm = np.zeros((num_classes, num_classes), dtype=np.int64)
np.add.at(cm, (labels, preds), 1)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="viridis")
ax.set_xlabel("predicted class index")
ax.set_ylabel("true class index")
ax.set_title("Confusion matrix (196 classes) - bright diagonal = correct")
fig.colorbar(im, ax=ax, label="count")
plt.tight_layout()
plt.show()

In [ ]:
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
per_class_total = cm.sum(axis=1)

pairs = [(i, j, int(cm_off[i, j])) for i in range(num_classes) for j in range(num_classes) if cm_off[i, j] > 0]
pairs.sort(key=lambda p: -p[2])

print("most confused class pairs (true -> predicted):")
print()
for i, j, count in pairs[:10]:
    share = count / max(per_class_total[i], 1) * 100.0
    gt   = ckpt["classes"][i]
    pred = ckpt["classes"][j]
    print("  %3d -> %3d | %5d imgs | %4.1f%% of class | %s -> %s"
          % (i, j, count, share, textwrap.shorten(gt, 40), textwrap.shorten(pred, 40)))

## 6. Per-class metrics

Precision, recall and F1 per class, derived directly from the confusion matrix, with the
macro average over all 196 classes. The extremes are the interesting part: the weakest
classes show where the baseline fails (useful context when judging the distilled model
later).

In [ ]:
tp = np.diag(cm).astype(float)
fp = cm.sum(axis=0) - tp    # predicted as this class but true class differs
fn = cm.sum(axis=1) - tp    # true class but predicted as something else

precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
recall    = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
f1        = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision),
                      where=(precision + recall) > 0)

print("macro averages over %d classes: precision %.3f | recall %.3f | f1 %.3f"
      % (num_classes, precision.mean(), recall.mean(), f1.mean()))
print()

order = np.argsort(f1)
print("weakest 15 classes (by F1):")
for i in order[:15]:
    print("  f1 %5.2f | precision %5.2f | recall %5.2f | %3d imgs | %s"
          % (f1[i], precision[i], recall[i], per_class_total[i], textwrap.shorten(ckpt["classes"][i], 45)))
print()
print("strongest 15 classes (by F1):")
for i in order[::-1][:15]:
    print("  f1 %5.2f | precision %5.2f | recall %5.2f | %3d imgs | %s"
          % (f1[i], precision[i], recall[i], per_class_total[i], textwrap.shorten(ckpt["classes"][i], 45)))

## 7. Model size and latency

Basic deployment-relevant facts about the baseline. Latency here is a rough indicator on
this machine only - the authoritative comparison happens on the Jetson Orin Nano in a
later phase, but measuring with an identical protocol from now on keeps the numbers
comparable.

In [ ]:
def measure_latency(model, device, iters, warmup, image_size):
    """Mean forward-pass time per single image, in milliseconds."""
    x = torch.randn(1, 3, image_size, image_size, device=device)
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000.0


param_total = sum(p.numel() for p in model.parameters())
size_on_disk_mb = CHECKPOINT_PATH.stat().st_size / 1e6
fp32_mb = param_total * 4 / 1e6

print("architecture        :", ckpt["model_arch"])
print("parameters          : %.2f M" % (param_total / 1e6))
print("weights (fp32)      : %.1f MB" % fp32_mb)
print("checkpoint on disk  : %.1f MB (includes metadata + history)" % size_on_disk_mb)
print()

lat_gpu = measure_latency(model, DEVICE, LATENCY_ITERS_GPU, LATENCY_WARMUP, ckpt["preprocessing"]["image_size"])
print("latency @ %s, batch=1 : %6.2f ms/img  (~%5.1f FPS)"
      % (str(DEVICE).upper(), lat_gpu, 1000.0 / lat_gpu))
if DEVICE.type == "cuda":
    lat_cpu = measure_latency(model, torch.device("cpu"), LATENCY_ITERS_CPU, LATENCY_WARMUP,
                              ckpt["preprocessing"]["image_size"])
    print("latency @ CPU, batch=1 : %6.2f ms/img  (~%5.1f FPS)  [rough edge proxy]"
          % (lat_cpu, 1000.0 / lat_cpu))

## 8. Representative predictions

The most **confident correct** predictions and the most **confident mistakes**. Confident
errors are the interesting ones for a fine-grained task like this - usually visually
similar models (e.g. two sedan years of the same brand).

In [ ]:
correct_mask = preds == labels

def show_examples(indices, title):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, i in zip(axes.ravel(), indices):
        path, true_label = data["test"][i]
        pred_label = int(preds[i])
        ok = bool(correct_mask[i])
        ax.imshow(Image.open(path))
        title_gt = textwrap.shorten(ckpt["classes"][true_label], 28)
        title_pred = textwrap.shorten(ckpt["classes"][pred_label], 28)
        ax.set_title(("OK    " if ok else "WRONG\n") + "gt:   " + title_gt
                     + "\npred: " + title_pred + " (%.1f%%)" % (confs[i] * 100),
                     fontsize=8, color="green" if ok else "red")
        ax.axis("off")
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


best_correct = np.argsort(-np.where(correct_mask, confs, -1.0))[:8]
show_examples(best_correct, "Most confident correct predictions")

In [ ]:
best_wrong = np.argsort(-np.where(~correct_mask, confs, -1.0))[:8]
show_examples(best_wrong, "Most confident incorrect predictions")
print("incorrect predictions: %d / %d (%.1f%%)" % ((~correct_mask).sum(), n, (~correct_mask).mean() * 100))

## 9. Summary

The baseline numbers produced above are the reference for the rest of the project:

- **Accuracy** - test top-1 / top-5, macro precision/recall/F1 (sections 4-6)
- **Size** - parameters and checkpoint MB (section 7)
- **Speed** - batch=1 latency with the protocol in section 7

Next: `03_video_inference.ipynb` wraps this classifier into the video pipeline
(detector -> crop -> classify -> annotate). Later phases (distilled student, TensorRT,
Jetson) will rerun this notebook's measurements against their own artifacts.